# Fair Compensation Project

### Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import ast
import joblib

df = pd.read_csv('data/cleaned/modelling.csv')
print(f"Starting shape: {df.shape}")

Starting shape: (36450, 34)


In [2]:
df['fairness_index'] = df['annual_mean'] / df['bls_annual_mean']

def label_fairness(a):
    if pd.isna(a):
        return np.nan
    elif a < 0.9:
        return 'underpaid'
    elif a <= 1.1:
        return 'fair'
    else:
        return 'generous'

df['fairness_label'] = df['fairness_index'].apply(label_fairness)
print("Fairness Index Stats:")
print(df['fairness_index'].describe().round(3))
print()
print("Fairness Label Count:")
print(df['fairness_label'].value_counts())
print()
print("As percentages:")
print(df['fairness_label'].value_counts(normalize = True).round(3) * 100)
        

Fairness Index Stats:
count    36450.000
mean         0.890
std          0.344
min          0.108
25%          0.650
50%          0.831
75%          1.070
max          4.116
Name: fairness_index, dtype: float64

Fairness Label Count:
fairness_label
underpaid    21346
generous      8241
fair          6863
Name: count, dtype: int64

As percentages:
fairness_label
underpaid    58.6
generous     22.6
fair         18.8
Name: proportion, dtype: float64


In [3]:
# check fairness by occupation group
print(df.groupby('jobs_group')['fairness_index'].agg(['mean', 'median', 'count']).round(3))
print()
# check fairness by state
print(df.groupby('state_x')['fairness_index'].median().sort_values().head(10))
print()
# check bls
print(df[['jobs_group', 'annual_mean', 'bls_annual_mean', 'fairness_index']].groupby('jobs_group').mean().round(0))

                        mean  median  count
jobs_group                                 
Analyst                0.735   0.686   2204
Business Analyst       0.863   0.826   5592
Business Intelligence  1.131   1.073   1796
CFO                    0.798   0.750   2416
Controller             1.164   1.117   4776
Data Analyst           0.736   0.686   4233
Data Engineer          0.936   0.901   2919
Data Scientist         1.121   1.089   2295
Finance                0.793   0.780   2515
Financial Analyst      0.773   0.733   6472
Operations Analyst     0.868   0.812   1232

state_x
VT    0.623159
MT    0.660348
NY    0.733341
SC    0.751874
SD    0.764439
WI    0.770936
MA    0.781541
GA    0.782552
NJ    0.793467
CO    0.804391
Name: fairness_index, dtype: float64

                       annual_mean  bls_annual_mean  fairness_index
jobs_group                                                         
Analyst                    81471.0         111867.0             1.0
Business Analyst           

In [4]:
cols_to_drop = [
    'job_title', 'company', 'company_clean', 'company_co', 'city', 'state_x', 'state_y', 'state_abbr', 'state_name', 'bls_occ_code', 'revenue_band', 'employee_band', 'sector', 'bls_median_wage',
    'revenue_rank', 'employee_rank', 'company_score', 'reviews_count',
]

cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns = cols_to_drop)
print(f"SHape after dropping columns: {df.shape}")

SHape after dropping columns: (36450, 19)


In [5]:
from collections import Counter

df['total_employed'] = pd.to_numeric(df['total_employed'].astype(str).str.replace(',', ''), errors = 'coerce')

cat_cols = [
    'jobs_group', 'seniority', 'remote', 'occ_title', 'sector_group'
]

df = pd.get_dummies(df, columns = cat_cols, prefix = cat_cols, dummy_na = False)

print(f"Shape after one-hot encoding category columns: {df.shape}")

def parse_skills(a):
    try:
        result = ast.literal_eval(a)
        return result if isinstance(result, list) else []
    except:
        return []
df['skills_parsed'] = df['skills'].apply(parse_skills)
all_skills = [s for sublist in df['skills_parsed'] for s in sublist]
top_skills = [skill for skill, count in Counter(all_skills).most_common(30)]
print("Top 30 Skills:")
print(top_skills)

for skill in top_skills:
    col_name = f'skill_{skill.lower().replace(" ", "_").replace(".", "")}'
    df[col_name] = df['skills_parsed'].apply(lambda x: int(skill in x))
df = df.drop(columns = ['skills', 'skills_parsed'])
print(f"Shape after dealing with skills columns: {df.shape}")



Shape after one-hot encoding category columns: (36450, 67)
Top 30 Skills:
['Bachelor', 'Office', 'Excel', 'SQL', 'Master', 'Python', 'CPA', 'Word', 'PowerPoint', 'Tableau', 'Power BI', 'Access', 'ERP', 'SAP', 'Agile', 'AWS', 'R', 'Oracle', 'MBA', 'English', 'Azure', 'Spark', 'Artificial Intelligence', 'Java', 'Salesforce', 'Machine Learning', 'Hadoop', 'HTML', 'PhD', 'CMA']
Shape after dealing with skills columns: (36450, 96)


In [6]:
df.to_csv('data/cleaned/modelling_features.csv', index = False)
print(f"Saved modelling_features.csv: {df.shape}")
# print(f"\nFinal columns ({len(df.columns)}):")
# print(df.columns.tolist())
# print(f"\nFairness label distribution:")
# print(df['fairness_label'].value_counts())

Saved modelling_features.csv: (36450, 96)
